# Graph coloring on the Washington map

*Generated by `utils/make_notebooks.py`. Edit the template there, not this file.*

This notebook uses `utils/color.py` to turn Washington's counties (and congressional districts) into an
adjacency graph, then shows how each coloring algorithm works on it:

1. Load the map and build the dual graph
2. Compare the heuristic colorings with the exact minimum coloring (backtracking and SAT)
3. Animate greedy / DSatur step by step
4. Animate the exact backtracking search, then cross-check it with a SAT solver
5. Show why the lower bound holds, and how much vertex order matters
6. Show how the boundary source (TIGER vs. cartographic) changes the graph
7. Congressional districts, plus which counties are split between districts

**Requirements:** `pip install geopandas networkx matplotlib requests pandas python-sat`. The first run downloads
Census shapefiles into `./data/` (the national TIGER county file is about 80 MB). Later runs read
from that cache.

Nodes are relabeled to integers with `wc.integer_labels`, the same as in `bench.py`, so the color
counts here match `results/colorings.csv` and don't change between kernel restarts.

In [ ]:
import os, random, sys, time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from matplotlib import animation
from matplotlib.patches import Patch
from IPython.display import HTML, display
from pysat.solvers import Solver  # imported here so section 2's SAT timing excludes import cost

# Run from the repo root so `utils`, data/, out/ and results/ resolve (see CLAUDE.md).
if Path.cwd().name == "stateNotebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # ahead of any site-packages `utils`
import utils.color as wc

STATE = "WA"
plt.rcParams["figure.dpi"] = 110
plt.rcParams["animation.embed_limit"] = 100  # MB, for the inline animations
UNCOLORED = "#e6e6e6"
SIMPLIFY_M = 250       # simplify outlines for drawing only (graphs use full geometry)
MAX_FRAMES = 80        # longer animations are subsampled

## 1. Load the counties and build the graph

`SOURCE` selects the boundary file:

* `"tiger"`: legal boundaries, which extend into water (bays, lakes, coastal waters). Counties can
  touch across water.
* `"cb"`: cartographic boundaries clipped to the shoreline. Borders that exist only over water
  disappear, and islands may become isolated nodes.

`MODE = "rook"` is the standard map-coloring rule: two regions conflict only if they share a border
segment, not just a corner point.

In [ ]:
LAYER, SOURCE, MODE = "county", "tiger", "rook"
MIN_SHARED_M, SNAP_M = 100.0, 0.0

cfg = wc.SOURCES[LAYER]
ID, NAME = cfg["id_col"], cfg["name_col"]


def load(layer, source=SOURCE, mode=MODE):
    """Fetch a layer, build its graph with integer labels, and tag each row with its node."""
    c = wc.SOURCES[layer]
    g = wc.fetch_layer(layer, source, STATE)
    G = wc.integer_labels(wc.build_graph(g, c["id_col"], c["name_col"], mode, MIN_SHARED_M, SNAP_M))
    g["node"] = g[c["id_col"]].map({d["geoid"]: n for n, d in G.nodes(data=True)})
    g["geometry"] = g.simplify(SIMPLIFY_M)
    return g, G


gdf, G = load(LAYER)
names = nx.get_node_attributes(G, "name")
pos = {n: (d["x"], d["y"]) for n, d in G.nodes(data=True)}
LABELS = G.number_of_nodes() <= 80  # county names are unreadable on bigger maps

print(f"{STATE}: {G.number_of_nodes()} regions, {G.number_of_edges()} shared borders")
print("planar:", nx.check_planarity(G)[0], "| max degree:", max(d for _, d in G.degree))
print("isolated:", [names[n] for n in nx.isolates(G)] or "none")

In [ ]:
def paint(coloring, ax, title="", highlight=(), edge_hl=(), g=None, graph=None):
    """Fill each region with its color class; regions missing from `coloring` are gray."""
    g = gdf if g is None else g
    graph = G if graph is None else graph
    fills = g["node"].map(
        lambda n: wc.PALETTE[coloring[n] % len(wc.PALETTE)] if n in coloring else UNCOLORED
    )
    g.plot(color=fills, edgecolor="white", linewidth=0.6, ax=ax)
    if highlight:
        g[g["node"].isin(highlight)].boundary.plot(ax=ax, color="black", linewidth=2)
    if edge_hl:
        p = {n: (d["x"], d["y"]) for n, d in graph.nodes(data=True)}
        nx.draw_networkx_edges(graph, p, edgelist=list(edge_hl), ax=ax, width=2.5, edge_color="black")
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()


def legend(k, ax, loc="lower left"):
    ax.legend(handles=[Patch(color=wc.PALETTE[c], label=f"color {c}") for c in range(k)],
              loc=loc, fontsize=7, frameon=False)

### The dual graph: one node per county, one edge per shared border

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
gdf.plot(color="#f2f2f2", edgecolor="#9a9a9a", linewidth=0.5, ax=ax)
nx.draw_networkx_edges(G, pos, ax=ax, width=0.9, edge_color="#444444")
deg = dict(G.degree)
scale = 1 if LABELS else 0.35
nodes = nx.draw_networkx_nodes(G, pos, ax=ax, node_size=[scale * (60 + 25 * deg[n]) for n in G],
                               node_color=[deg[n] for n in G], cmap="viridis")
if LABELS:
    nx.draw_networkx_labels(G, pos, labels=names, ax=ax, font_size=6)
fig.colorbar(nodes, ax=ax, shrink=0.6, label="degree (number of neighbours)")
ax.set_title(f"{STATE} {LAYER} adjacency graph ({SOURCE}, {MODE})")
ax.set_axis_off()
plt.show()

## 2. Every algorithm side by side

In [ ]:
results, timings = {}, {}
for algo, strategy in wc.ALGORITHMS.items():
    t0 = time.perf_counter()
    col = nx.greedy_color(G, strategy=strategy)
    timings[algo] = time.perf_counter() - t0
    results[algo] = {"coloring": col, "k": len(set(col.values()))}

t0 = time.perf_counter()
col, k = wc.exact_coloring(G)
timings["exact"] = time.perf_counter() - t0
results["exact"] = {"coloring": col, "k": k}

t0 = time.perf_counter()
col, k = wc.sat_coloring(G)
timings["sat"] = time.perf_counter() - t0
results["sat"] = {"coloring": col, "k": k}
assert results["sat"]["k"] == results["exact"]["k"], "exact and SAT disagree on the chromatic number"

for a, r in results.items():
    assert not wc.is_proper(G, r["coloring"]), f"{a} produced a conflict"

pd.DataFrame({
    "colors used": {a: r["k"] for a, r in results.items()},
    "time (ms)": {a: round(1000 * t, 2) for a, t in timings.items()},
    "class sizes": {a: sorted(Counter(r["coloring"].values()).values(), reverse=True)
                    for a, r in results.items()},
})

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (a, r) in zip(axes.flat, results.items()):
    paint(r["coloring"], ax, f"{a}: {r['k']} colors")
    legend(r["k"], ax)
for ax in axes.flat[len(results):]:
    ax.set_axis_off()
fig.suptitle(f"{STATE} {LAYER}: colorings by algorithm", fontsize=13)
plt.tight_layout()
plt.show()

Colorings can use the same number of colors and still be different maps. Here we check which
algorithms produce the **same partition** (identical color classes once you ignore which color is
which).

In [ ]:
parts = {a: wc.canonical_partition(r["coloring"]) for a, r in results.items()}
algos = list(parts)
pd.DataFrame([[parts[a] == parts[b] for b in algos] for a in algos], index=algos, columns=algos)

## 3. Step-by-step greedy coloring

Greedy coloring visits the vertices in some order and gives each one the **smallest color not
already used by a colored neighbour**. The strategy only changes the order:

* `largest_first`: highest degree first (a fixed order)
* `dsatur`: next vertex is the one with the most *distinct* neighbour colors (recomputed every step)
* `smallest_last`: repeatedly remove the min-degree vertex, then color in reverse removal order

`nx.greedy_color` fills its result dict in visit order, which lets us replay it. Animations longer
than `MAX_FRAMES` steps show every few steps.

In [ ]:
def greedy_frames(algo):
    col = results[algo]["coloring"]
    order = list(col)  # insertion order == visit order
    return [(dict((v, col[v]) for v in order[: i + 1]), order[i]) for i in range(len(order))]


def subsample(frames):
    stride = max(1, -(-len(frames) // MAX_FRAMES))
    idx = list(range(0, len(frames), stride))
    if idx[-1] != len(frames) - 1:
        idx.append(len(frames) - 1)
    return idx


def animate(frames, title_fn, interval=350):
    idx = subsample(frames)
    fig, ax = plt.subplots(figsize=(8.5, 6))
    def draw(j):
        ax.clear()
        coloring, current = frames[idx[j]][:2]
        paint(coloring, ax, title_fn(idx[j]), highlight=[current])
    anim = animation.FuncAnimation(fig, draw, frames=len(idx), interval=interval)
    plt.close(fig)
    return HTML(anim.to_jshtml())


ALGO = "dsatur"  # try "largest_first", "smallest_last", "independent_set"
frames = greedy_frames(ALGO)
animate(frames, lambda i: f"{ALGO} step {i + 1}/{len(frames)}: "
                          f"{names[frames[i][1]]} gets color {frames[i][0][frames[i][1]]}")

A static version for reports. The visit order is printed on each county (on small maps only), so
you can see where the algorithm starts and how it spreads.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, algo in zip(axes, ["largest_first", "dsatur", "smallest_last"]):
    col = results[algo]["coloring"]
    paint(col, ax, f"{algo} ({results[algo]['k']} colors)" + (", numbers = visit order" if LABELS else ""))
    if LABELS:
        for step, v in enumerate(col, 1):
            ax.annotate(str(step), pos[v], ha="center", va="center", fontsize=6.5, weight="bold")
plt.tight_layout()
plt.show()

## 4. The exact search (backtracking)

`wc.exact_coloring` tries k = (max clique size), k+1, … and for each k runs a DSatur-ordered
backtracking search. The function below is the same search as `wc._try_k` with **tracing added**,
so we can watch it assign colors and undo them.

In [ ]:
def traced_try_k(G, k, max_events=5000):
    colors, events = {}, []

    def pick():
        best, key = None, None
        for v in G:
            if v in colors:
                continue
            sat = len({colors[u] for u in G[v] if u in colors})
            cand = (sat, G.degree(v))
            if key is None or cand > key:
                best, key = v, cand
        return best

    def rec():
        v = pick()
        if v is None:
            return True
        used = {colors[u] for u in G[v] if u in colors}
        max_open = max(colors.values(), default=-1) + 1
        for c in range(min(k, max_open + 1)):
            if c not in used:
                colors[v] = c
                if len(events) < max_events:
                    events.append((dict(colors), v, "assign"))
                if rec():
                    return True
                del colors[v]
                if len(events) < max_events:
                    events.append((dict(colors), v, "undo"))
        return False

    ok = rec()
    return (dict(colors) if ok else None), events


lower = max(len(c) for c in nx.find_cliques(G))
traces = {}
for kk in range(lower, results["exact"]["k"] + 1):
    sol, ev = traced_try_k(G, kk)
    traces[kk] = ev
    undos = sum(e[2] == "undo" for e in ev)
    print(f"k={kk}: {'solved' if sol else 'impossible'} after {len(ev)} recorded steps ({undos} undos)")

In [ ]:
K_SHOW = results["exact"]["k"]  # or `lower` to watch a failing search
ev = traces[K_SHOW]
animate(ev, lambda i: f"backtracking, k={K_SHOW}: step {i + 1}/{len(ev)} {ev[i][2]} {names[ev[i][1]]}",
        interval=200)

### Cross-check: the same question as a SAT problem

`wc.sat_coloring` answers "can this graph be colored with k colors?" with a SAT solver (Glucose 4
via `python-sat`) instead of a hand-written search:

* one boolean variable x[v, c] per region v and color c, meaning "v gets color c"
* for every region, a clause x[v, 0] ∨ … ∨ x[v, k-1] (it gets at least one color)
* for every border (u, w) and color c, a clause ¬x[u, c] ∨ ¬x[w, c] (neighbours differ)
* one maximum clique is fixed to colors 0, 1, …, so the solver doesn't try renamed copies of the
  same coloring

Like `exact_coloring`, it tries k = clique size, k+1, … and stops at the first satisfiable k. It
shares no search code with the backtracking, so when the two agree, that is good evidence the
chromatic number is right. The table shows the size of each formula and the solver's answer.

In [ ]:
lower = len(max(nx.find_cliques(G), key=len))
rows = []
for kk in range(lower, results["sat"]["k"] + 1):
    var = {(v, c): i * kk + c + 1 for i, v in enumerate(G) for c in range(kk)}
    clauses = [[var[v, c] for c in range(kk)] for v in G]
    clauses += [[-var[u, c], -var[w, c]] for u, w in G.edges for c in range(kk)]
    t0 = time.perf_counter()
    with Solver(name="glucose4", bootstrap_with=clauses) as s:
        ok = s.solve()  # no clique pinning here, to show the plain encoding
        stats = s.accum_stats()
    rows.append({"k": kk, "variables": len(var), "clauses": len(clauses),
                 "answer": "SAT" if ok else "UNSAT", "conflicts": stats.get("conflicts"),
                 "time (ms)": round(1000 * (time.perf_counter() - t0), 2)})

print(f"chromatic number: backtracking {results['exact']['k']}, SAT {results['sat']['k']}")
pd.DataFrame(rows).set_index("k")

The two optimal colorings usually differ even though they use the same number of colors. Outlined
regions are colored differently once colors are matched up as well as possible.

In [ ]:
from itertools import permutations

a, b = results["exact"]["coloring"], results["sat"]["coloring"]
k = results["exact"]["k"]
# rename SAT's colors to best match the backtracking colors (k! options, k is small here)
best = max(permutations(range(k)), key=lambda p: sum(a[v] == p[b[v]] for v in G))
b = {v: best[c] for v, c in b.items()}
differ = [v for v in G if a[v] != b[v]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
paint(a, axes[0], f"backtracking: {k} colors")
paint(b, axes[1], f"SAT: {k} colors, {len(differ)} of {G.number_of_nodes()} regions differ",
      highlight=differ)
legend(k, axes[0])
plt.tight_layout()
plt.show()

## 5. Why can't we use fewer colors?

The **maximum clique** gives a lower bound: regions that all touch each other need distinct colors.
If the exact search needs more colors than the clique size, the obstruction is something else,
usually an **odd wheel** (a hub surrounded by an odd cycle of neighbours). The cycle needs 3 colors
and the hub needs a 4th.

In [ ]:
clique = max(nx.find_cliques(G), key=len)
chi = results["exact"]["k"]
print(f"max clique ({len(clique)}):", [names[n] for n in clique], f"| chromatic number: {chi}")

def odd_wheels(G):
    """Hubs whose neighbourhood contains an odd cycle (so hub + cycle needs 4 colors)."""
    for hub in G:
        H = G.subgraph(G[hub])
        if not nx.is_bipartite(H):
            cyc = next(c for c in nx.simple_cycles(H) if len(c) % 2 == 1)
            yield hub, cyc

wheel = next(odd_wheels(G), None) if chi > len(clique) else None

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
paint(results["exact"]["coloring"], axes[0], f"max clique of size {len(clique)} (outlined)",
      highlight=clique)
if wheel:
    hub, cyc = wheel
    paint(results["exact"]["coloring"], axes[1],
          f"odd wheel: hub {names[hub]} + {len(cyc)}-cycle", highlight=[hub, *cyc],
          edge_hl=[(hub, c) for c in cyc] + list(zip(cyc, cyc[1:] + cyc[:1])))
else:
    axes[1].set_axis_off()
    axes[1].set_title("clique bound equals chromatic number: no odd wheel needed")
plt.tight_layout()
plt.show()

### Vertex order matters

Plain greedy with a **random** vertex order. Every run is a valid coloring, but the number of
colors depends on the order. The ordering heuristics above exist to steer away from the bad orders.

In [ ]:
rng = random.Random(0)
counts = Counter()
worst = None
for _ in range(2000):
    order = list(G)
    rng.shuffle(order)
    col = nx.greedy_color(G, strategy=lambda _G, _c, o=order: iter(o))
    k = len(set(col.values()))
    counts[k] += 1
    if worst is None or k > worst[0]:
        worst = (k, col)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1, 1.6]})
ks = sorted(counts)
a1.bar([str(k) for k in ks], [counts[k] for k in ks], color="#4C78A8")
a1.set_xlabel("colors used"); a1.set_ylabel("runs (of 2000)")
a1.set_title("greedy with random vertex orders")
paint(worst[1], a2, f"worst random order found: {worst[0]} colors")
legend(worst[0], a2)
plt.tight_layout()
plt.show()

## 6. How the boundary data changes the graph

The same state gives different graphs depending on the boundary file and the adjacency rule.
Colored edges are present in one version and missing from the other. Integer labels come from
sorted county IDs, so node `i` is the same county in every version.

In [ ]:
variants = {
    "tiger / rook": G if (SOURCE, MODE) == ("tiger", "rook") else load(LAYER, "tiger", "rook")[1],
    "cb / rook": load(LAYER, "cb", "rook")[1],
    "tiger / queen": load(LAYER, "tiger", "queen")[1],
}
assert all(nx.get_node_attributes(g, "geoid") == nx.get_node_attributes(G, "geoid")
           for g in variants.values()), "variants cover different counties"

def edge_set(g):
    return {frozenset(e) for e in g.edges}

base = edge_set(variants["tiger / rook"])

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, name in zip(axes, ["cb / rook", "tiger / queen"]):
    other = edge_set(variants[name])
    only_base, only_other = base - other, other - base
    gdf.plot(color="#f2f2f2", edgecolor="#9a9a9a", linewidth=0.4, ax=ax)
    nx.draw_networkx_edges(G, pos, ax=ax, width=0.6, edge_color="#bbbbbb")
    nx.draw_networkx_edges(G, pos, edgelist=[tuple(e) for e in only_base], ax=ax,
                           width=2.5, edge_color="#E45756", label="only in tiger / rook")
    nx.draw_networkx_edges(G, pos, edgelist=[tuple(e) for e in only_other], ax=ax,
                           width=2.5, edge_color="#4C78A8", style="dashed", label=f"only in {name}")
    ax.set_title(f"tiger / rook vs {name}: -{len(only_base)} / +{len(only_other)} edges")
    if only_base or only_other:
        ax.legend(loc="lower left", fontsize=8, frameon=False)
    ax.set_axis_off()
    for e in sorted(only_base | only_other, key=sorted):
        print(f"{name:>14}: {'missing' if e in only_base else 'extra  '} "
              + " - ".join(names[n] for n in sorted(e)))
plt.tight_layout()
plt.show()

pd.DataFrame({name: {"edges": g.number_of_edges(),
                     "isolated": len(list(nx.isolates(g))),
                     "chromatic number": wc.exact_coloring(g)[1],
                     "chromatic number (SAT)": wc.sat_coloring(g)[1]}
              for name, g in variants.items()}).T

## 7. Congressional districts

In [ ]:
cd_gdf, G_cd = load("cd")
cd_col, cd_k = wc.exact_coloring(G_cd)
assert wc.sat_coloring(G_cd)[1] == cd_k, "exact and SAT disagree on the district graph"
cd_names = nx.get_node_attributes(G_cd, "name")
cd_pos = {n: (d["x"], d["y"]) for n, d in G_cd.nodes(data=True)}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 6))
paint(cd_col, a1, f"{STATE} congressional districts: exact coloring, {cd_k} colors",
      g=cd_gdf, graph=G_cd)
for n, (x, y) in cd_pos.items():
    a1.annotate(cd_names[n].removeprefix("CD-"), (x, y), ha="center", fontsize=6, weight="bold")

cd_gdf.plot(color="#f2f2f2", edgecolor="#9a9a9a", linewidth=0.5, ax=a2)
nx.draw_networkx(G_cd, cd_pos, ax=a2, node_color=[wc.PALETTE[cd_col[n]] for n in G_cd],
                 labels={n: s.removeprefix("CD-") for n, s in cd_names.items()},
                 font_size=6, node_size=180 if G_cd.number_of_nodes() > 20 else 500, width=0.8)
a2.set_title(f"district graph: {G_cd.number_of_nodes()} nodes, {G_cd.number_of_edges()} edges")
a2.set_axis_off()
plt.tight_layout()
plt.show()

Small urban districts are hard to see at state scale. This zooms in on the metro area around the
smallest district: that district plus its neighbours that are smaller than the median district.

In [ ]:
km2 = dict(zip(cd_gdf["node"], cd_gdf.area / 1e6))
seed = min(km2, key=km2.get)
cutoff = sorted(km2.values())[len(km2) // 2]
metro = [seed] + [n for n in G_cd[seed] if km2[n] < cutoff]
x0, y0, x1, y1 = cd_gdf[cd_gdf["node"].isin(metro)].total_bounds
pad = 0.15 * max(x1 - x0, y1 - y0)

fig, ax = plt.subplots(figsize=(9, 7))
paint(cd_col, ax, f"zoom: around {cd_names[seed]} (smallest district)", g=cd_gdf, graph=G_cd)
for n, (x, y) in cd_pos.items():
    if x0 - pad <= x <= x1 + pad and y0 - pad <= y <= y1 + pad:
        ax.annotate(cd_names[n], (x, y), ha="center", fontsize=7, weight="bold")
ax.set_xlim(x0 - pad, x1 + pad); ax.set_ylim(y0 - pad, y1 + pad)
plt.show()

### Counties split across districts

`wc.county_district_overlay` intersects the two layers (ignoring slivers under 1 km²) and lists
every county that falls in more than one district.

In [ ]:
split = wc.county_district_overlay(SOURCE, STATE)
print(f"{len(split)} of {G.number_of_nodes()} counties are split")
display(pd.DataFrame({"districts": {c: ", ".join(d) for c, d in split.items()}}))

fig, ax = plt.subplots(figsize=(10, 7.5))
gdf.plot(color=gdf[NAME].map(lambda n: "#F58518" if n in split else "#f2f2f2"),
         edgecolor="#9a9a9a", linewidth=0.4, ax=ax)
cd_gdf.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title(f"{len(split)} counties (orange) are split by district lines (black)")
ax.set_axis_off()
plt.show()

## 8. (Optional) Analyze a published map's coloring

If you transcribe the colors of a real county map into `{county name: color}`, `wc.analyze_reference`
checks that it is proper, compares it with the chromatic number, and tests whether plain greedy
could have produced it. The example below just recolors our DSatur result with color names;
replace it with your transcription.

In [ ]:
ref = {names[n]: ["blue", "orange", "green", "red", "purple", "brown", "teal"][c]
       for n, c in results["dsatur"]["coloring"].items()}
# ref = json.load(open("ref_colors.json"))
wc.analyze_reference(G, ref, results)